In [1]:
!pip install zss -q
!pip install antlr4-python3-runtime==4.11 -q

In [2]:
from kaggle_secrets import UserSecretsClient
secret_label = "GITHUB_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)

In [3]:
import os

token = secret_value

repo_url = f"https://{token}@github.com/smiling-demon/llm-symbolic-ode-reasoning-dev.git"
repo_dir = "llm-symbolic-ode-reasoning-dev"

if not os.path.exists(repo_dir):
    !git clone {repo_url}

os.chdir(repo_dir)

!pwd

Cloning into 'llm-symbolic-ode-reasoning-dev'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 135 (delta 54), reused 112 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 1.53 MiB | 23.70 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/kaggle/working/llm-symbolic-ode-reasoning-dev


In [4]:
import pandas as pd

from llm import LLM
from methods import cot
from metrics import evaluate_predictions

In [5]:
df = pd.read_excel("/kaggle/input/datasets/dmitrylessy/ode-basic-dataset/data/test.xlsx")

equations = df['equation'].tolist()
solutions = df['solution'].tolist()

In [6]:
llm = LLM("Qwen/Qwen2.5-3B-Instruct")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
%%time
batch_size = 20
predictions = []

for i in range(0, len(equations), batch_size):
    predictions += cot(llm, equations[i: i + batch_size], max_new_tokens=1024)

CPU times: user 8min 32s, sys: 1.65 s, total: 8min 33s
Wall time: 8min 34s


In [8]:
evaluation = evaluate_predictions(predictions, solutions)
for key in evaluation:
    evaluation[key] = [evaluation[key]]
pd.DataFrame.from_dict(evaluation)

,bleu,ast_error_size,ast_tree_similarity,exact_match
0,0.608449,0.523934,0.39946,0.44
